# Exercise 39: Movie Rating

**Source:** [LeetCode 1341](https://leetcode.com/problems/movie-rating/description/?envType=study-plan-v2&envId=top-sql-50) (Medium).

Table: `Movies`
```
+---------------+---------+
| Column Name   | Type    |
+---------------+---------+
| movie_id      | int     |
| title         | varchar |
+---------------+---------+
```
- `movie_id` is the primary key.
- `title` is the name of the movie (each movie has a unique title).

Table: `Users`
```
+---------------+---------+
| Column Name   | Type    |
+---------------+---------+
| user_id       | int     |
| name          | varchar |
+---------------+---------+
```
- `user_id` is the primary key.
- `name` is the name of the `user_id`.

Table: `MovieRating`
```
+---------------+---------+
| Column Name   | Type    |
+---------------+---------+
| movie_id      | int     |
| user_id       | int     |
| rating        | int     |
| created_at    | date    |
+---------------+---------+
```
- `(movie_id, user_id)` is the primary key.
- Together with the movie and the user is the user's `rating` of a movie.
- `created_at` is the user's review date.

**Task.**
Return two results:
1) Find the user with the highest number of ratings. Return the lexicographically smaller user name in case of a tie.
2) Find the movie with the highest average rating in February 2020. Return the lexicographically smaller movie in case of a tie.

Connect to the database `postgres` using the username `postgres`.

In [1]:
%load_ext sql
%sql postgresql+psycopg://postgres@localhost:5432/postgres

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Connecting to 'postgresql+psycopg://postgres@localhost:5432/postgres'

### Create new database for the exercise

Make sure first that a database with the same name is removed in case it already exists.

In [2]:
%sql DROP DATABASE IF EXISTS exercise_39 WITH (FORCE);
%sql CREATE DATABASE exercise_39;

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

Connect to the database.

*Caution.* No semicolon at the end otherwise it belongs to the name of the database.

In [3]:
%sql postgresql+psycopg://postgres@localhost:5432/exercise_39 --alias exercise_39

Connecting and switching to connection 'exercise_39'

#### Fill the database with content

In [6]:
%%sql
-- Drop tables if already defined
DROP TABLE IF EXISTS MovieRating;
DROP TABLE IF EXISTS Users;
DROP TABLE IF EXISTS Movies;

-- Recreate the tables
CREATE TABLE Movies (
    movie_id INTEGER PRIMARY KEY,
    title VARCHAR UNIQUE NOT NULL
);
CREATE TABLE Users (
    user_id INTEGER PRIMARY KEY,
    name VARCHAR NOT NULL
);
CREATE TABLE MovieRating (
    movie_id INTEGER NOT NULL,
    user_id INTEGER NOT NULL,
    rating INTEGER,
    created_at DATE,
    PRIMARY KEY (movie_id, user_id),
    FOREIGN KEY (movie_id) REFERENCES Movies(movie_id),
    FOREIGN KEY (user_id) REFERENCES Users(user_id)
);
-- Insert rows into the tables
INSERT INTO Movies (movie_id, title)
VALUES
(1, 'Avengers'),
(2, 'Frozen 2'),
(3, 'Joker');

INSERT INTO Users (user_id, name)
VALUES
(1, 'Daniel'),
(2, 'Monica'),
(3, 'Maria'),
(4, 'James');

INSERT INTO MovieRating (movie_id, user_id, rating, created_at)
VALUES
(1, 1, 3,'2020-01-12'),
(1, 2, 4,'2020-02-11'),
(1, 3, 2,'2020-02-12'),
(1, 4, 1,'2020-01-01'),
(2, 1, 5,'2020-02-17'), 
(2, 2, 2,'2020-02-01'), 
(2, 3, 2,'2020-03-01'),
(3, 1, 3,'2020-02-22'), 
(3, 2, 4,'2020-02-25');

Running query in 'exercise_39'

3 rows affected.

4 rows affected.

9 rows affected.

++
||
++
++

#### Solve the exercise

**Task.**
Return two results:
1) Find the user with the highest number of ratings. Return the lexicographically smaller user name in case of a tie.
2) Find the movie with the highest average rating in February 2020. Return the lexicographically smaller movie in case of a tie.

In [ ]:
%%sql
(WITH rev_ratings AS (
    SELECT
        user_id, COUNT(CASE WHEN rating IS NOT NULL THEN 1 END) AS num_ratings
        FROM
            MovieRating
        GROUP BY
            user_id
)
SELECT
    U.name AS results
FROM Users U INNER JOIN rev_ratings R ON U.user_id = R.user_id
WHERE
    R.num_ratings = (SELECT MAX(num_ratings) FROM rev_ratings) ORDER BY name LIMIT 1
)
UNION ALL
(WITH AvgRatings AS (
    SELECT movie_id, AVG(rating) AS avg_rating FROM MovieRating
    WHERE created_at >= '2020-02-01' AND created_at < '2020-03-01'
    GROUP BY movie_id
)

SELECT M.title AS results FROM Movies M INNER JOIN AvgRatings A ON M.movie_id = A.movie_id
WHERE
    A.avg_rating = (SELECT MAX(avg_rating) FROM AvgRatings) ORDER BY results LIMIT 1
)

Running query in 'exercise_39'

2 rows affected.

results
Daniel
Frozen 2


#### Explanation

- Do the two tasks separately. Save both under the column `results` and then *join the rows* using `UNION ALL`.
- We do both tasks using an auxiliary table `WITH ... AS ([define table])`
- For the *first task* count for every `user_id` the number of ratings using `GROUP BY`. Then perform an `INNER JOIN` on the `user_id` with `Users` to find the `name` of the user with the most ratings. To find the `MAX` ratings to a subquery in the `WHERE` clause.
- For the *second task* take the average rating `AVG(rating)` over all movie ratings in February. Then perform an `INNER JOIN` with `Movies` on the `movie_id` to obtain the `title` of the film.
- For both tasks in order to have only the lexicographically smallest output in case of a tie perform `ORDER BY results LIMIT 1`.

#### Close connection

In [8]:
%sql --close exercise_39
%sql postgresql+psycopg://postgres@localhost:5432/postgres
%sql DROP DATABASE IF EXISTS exercise_39;

Switching to connection 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

**Well done!** (approved by Leet)